In [1]:
import numpy as np
import scipy.sparse as sp
import time
from dolphindes.cvxopt import DenseSharedProjQCQP, OptimizationHyperparameters

# conda clean --all

In [2]:
# Physical constants
epsilon_0 = 8.854e-12 # F/m
mu_0 = 4e-7 * np.pi # H/m
c = 1/np.sqrt(epsilon_0 * mu_0) # speed of light in vacuum

E0 = 1e3 # plane wave amplitude in V/m

frequency = 2.45e9 # frequency in Hz
wavelength = c / frequency # wavelength in m
print(f"Wavelength: {wavelength} m")

ka = 1 # electrical size of the antenna
k = 2 * np.pi / wavelength # wavenumber
a = ka / k # antenna circumradius
print(f"Antenna circumradius: {a} m")

omega = 2 * np.pi * frequency # angular frequency

conductivity_reduction_factor = 1 # factor to reduce conductivity for testing purposes
copper_conductivity = conductivity_reduction_factor*5.96e7 # S/m
copper_permittivity = 1 + 1j * copper_conductivity / (omega * epsilon_0) # copper permittivity in dimensionless units
print(f"Copper permittivity: {copper_permittivity:.2e}")

Wavelength: 0.1223655664053944 m
Antenna circumradius: 0.019475084757658086 m
Copper permittivity: 1.00e+00+4.37e+08j


In [3]:
# Calculate surface impedance
delta = np.sqrt(2 / (omega * mu_0 * copper_conductivity)) # skin depth in m
Zs = (1 + 1j) / (copper_conductivity * delta)
# Zs = 1 / (copper_conductivity * delta)
print(f"Surface impedance: {Zs} Ω")

# Load matrices from text files
Lmat = np.loadtxt(r"Lmat_matrix.txt", delimiter=',') # lossy matrix
R0 = np.loadtxt(r"R0_matrix.txt", delimiter=',') # radiated power matrix
X0 = np.loadtxt(r"X0_matrix.txt", delimiter=',') # reactive power matrix
Vinc = E0*np.loadtxt(r"V_vector.txt", delimiter=',') # voltage excitation vector
Zmat = Zs * Lmat # material impedance matrix
Z0 = R0 + 1j*X0 # free-space impedance matrix
Ztot = Z0 + Zmat # total impedance matrix
Rmat = np.real(Zmat) # rezistivity matrix

Ndes = Lmat.shape[0] # number of design variables
print("Number of design variables: ", Ndes)

Vzero = np.zeros((Ndes,), dtype=complex) # zero vector

Obj = -0.5 * R0
obj = Vzero/2
obj0 = 0

# full plate performance
Ifull = np.linalg.solve(Ztot, Vinc) # current distribution for full plate
Pfull = -np.real(Ifull.conj().T @ Obj @ Ifull) + 2*np.real(Ifull.conj().T @ obj) + obj0 # objective for full plate
print(f"Objective for full plate: {Pfull:.6e} W")

Surface impedance: (0.012739130327240646+0.012739130327240646j) Ω
Number of design variables:  195
Objective for full plate: 1.010821e+01 W


In [4]:
# Assuming Ndes is OP.BF.nUnknowns and is an odd integer
k = (Ndes - 1) // 2

# 1. Upper block: Identity matrix of size (k + 1)
upper = np.eye(k + 1)

# 2. Lower block: Flipped identity of size k, followed by a column of zeros
# np.fliplr(np.eye(k)) creates the anti-diagonal matrix
lower_left = np.fliplr(np.eye(k))
lower_right = np.zeros((k, 1))
lower = np.hstack([lower_left, lower_right])

# 3. Vertically stack them to create C
C = np.vstack([upper, lower])
C = np.eye(Ndes) # skip transformation for testing purposes

# lossy matrix Cholesky factorization
Lchol = np.linalg.cholesky(C.conj().T @Lmat @ C).conj().T

# Mfactor = C @ np.linalg.inv(Lchol)
Mfactor = np.eye(Ndes) # skip transformation for testing purposes
# Mfactor = C

Vincf = Mfactor.conj().T @ Vinc
Ztotf = Mfactor.conj().T @ Ztot @ Mfactor
Z0f = Mfactor.conj().T @ Z0 @ Mfactor
Zmatf = Mfactor.conj().T @ Zmat @ Mfactor
Objf = Mfactor.conj().T @ Obj @ Mfactor
objf = Mfactor.conj().T @ obj
obj0f = obj0

Nfac = Z0f.shape[0]
iVec = np.zeros(Nfac, dtype=complex) # particular solution to satisfy the fixed current constraint (zero for now, can be used to satisfy a fixed current constraint by setting the iBFfixed row to np.linalg.solve(Z11, Vinc1) and the transformation matrix to have -np.linalg.solve(Z11, Z12) in the iBFfixed row)

In [5]:
# translate QCQP matrices to Dolphindes notation
Umat = 1j * Ztotf.conj() # Dolphindes U matrix
eVec = -1j*Vincf.conj() / 2 # Dolphindes e vector
Bmat = Objf # quadratic objective matrix
bVec = objf # linear objective vector
beta = obj0f # constant objective term

In [6]:
# preconditioner for projection constraints
G0 = Z0f # Green's function matrix
D = np.diag(np.diag(Z0f)) # Green's matrix diagonal
X = np.linalg.inv(Zmatf) # material admittance matrix
H0 = G0 - D
Y = np.linalg.solve(np.eye(Nfac) + X @ D, X)
iVec = X @ Vincf

S = np.eye(Nfac) + Y @ H0 # preconditioner matrix for projection constraints
q = -Y @ G0 @ iVec
p = q + S @ iVec

Sinv = np.linalg.inv(S)

P0 = Sinv @ Ztotf

sVec = Sinv @ q
tVec = sVec + iVec
Vtest = Ztotf @ tVec
print(f"voltage error: {np.max(np.abs(Vtest - Vincf))/np.max(np.abs(Vincf)):.2e}")

print(f"asymmetric error of preconditioner: {np.linalg.norm(S-S.T, ord='fro')/np.linalg.norm(S, ord='fro'):.2e}")

obj0p = obj0f - np.real(iVec.conj().T @ Objf @ iVec) + 2*np.real(iVec.conj().T @ objf)
objp = objf - Objf @ iVec
Objp = Objf

Ptr0 = -np.real(sVec.conj().T @ Objp @ sVec) + 2*np.real(sVec.conj().T @ objp) + obj0p
print(f"objective error: {np.abs(Ptr0 - Pfull)/np.abs(Pfull):.2e}")
print(f"constant term objective: {obj0p:.2e} W")


realPowerConstraint = np.real(-sVec.conj().T @ (-S) @ P0 @ sVec + 2*sVec.conj().T @ P0.conj().T @ (p/2) 
                              - sVec.conj().T @ (-S) @ P0 @ iVec - iVec.conj().T @ (-S) @ P0 @ sVec
                              - iVec.conj().T @ (-S) @ P0 @ iVec + 2*iVec.conj().T @ P0.conj().T @ (p/2))
realPower = np.real(tVec.conj().T @ S @ P0 @ tVec)

imagPowerConstraint = np.real(-sVec.conj().T @ (-S) @ (1j*P0) @ sVec + 2*sVec.conj().T @ (1j*P0).conj().T @ (-p/2) 
                              - sVec.conj().T @ (-S) @ (1j*P0) @ iVec - iVec.conj().T @ (-S) @ (1j*P0) @ sVec
                              - iVec.conj().T @ (-S) @ (1j*P0) @ iVec + 2*iVec.conj().T @ (1j*P0).conj().T @ (-p/2))
imagPower = np.real(tVec.conj().T @ S @ (1j*P0) @ tVec)

# print(f"3 elements of s1: {-p[:3]/2}")
Pv = np.column_stack((P0.conj().T @ -p/2, (1j*P0).conj().T @ -p/2))
print(f"3 rows of Pv: {Pv[:3,:]}")
Pv_precondL = np.column_stack((P0.conj().T @ -S.conj().T @ iVec, (1j*P0).conj().T @ -S.conj().T @ iVec))
print(f"3 rows of Pv_precondL: {Pv_precondL[:3,:]}")
FsL = Pv_precondL/2
print(f"3 rows of FsL: {FsL[:3,:]}")
Pv_precondR = np.column_stack((P0 @ iVec, (1j*P0) @ iVec))
print(f"3 rows of Pv_precondR: {Pv_precondR[:3,:]}")
FsR = - S @ Pv_precondR/2
print(f"3 rows of FsR: {FsR[:3,:]}")  # Debug print for the preconditioner contribution to the linear term
FsRL = FsR + FsL
print(f"3 rows of FsRL: {FsRL[:3,:]}")  # Debug print for the combined preconditioner contribution to the linear term
Fs = -Pv + FsRL
print(f"3 rows of Fs: {Fs[:3,:]}")


print(f"real power error: {np.abs(realPowerConstraint)/np.abs(realPower):.2e}")
print(f"imaginary power error: {np.abs(imagPowerConstraint)/np.abs(imagPower):.2e}")    

Umat = -S
eVec = -p/2
i0 = iVec 
Bmat = Objp
bVec = objp
beta = obj0p

voltage error: 1.36e-09
asymmetric error of preconditioner: 2.14e-09
objective error: 5.93e-11
constant term objective: 6.07e+04 W
3 rows of Pv: [[ 8.66481300e-05+8.74265163e-10j  8.74265163e-10-8.66481300e-05j]
 [ 8.66496734e-05-7.67286743e-10j -7.67286743e-10-8.66496734e-05j]
 [ 8.66495440e-05-6.29699740e-10j -6.29699740e-10-8.66495440e-05j]]
3 rows of Pv_precondL: [[-35.74751377-35.73135817j -35.73135817+35.74751377j]
 [ -6.8449807  -6.82879743j  -6.82879743 +6.8449807j ]
 [ -4.30359912 -4.28730727j  -4.28730727 +4.30359912j]]
3 rows of FsL: [[-17.87375688-17.86567909j -17.86567909+17.87375688j]
 [ -3.42249035 -3.41439871j  -3.41439871 +3.42249035j]
 [ -2.15179956 -2.14365363j  -2.14365363 +2.15179956j]]
3 rows of Pv_precondR: [[-65.15847695-65.15880883j  65.15880883-65.15847695j]
 [-58.57611975-58.57643197j  58.57643197-58.57611975j]
 [-59.24107351-59.24138666j  59.24138666-59.24107351j]]
3 rows of FsR: [[ 17.86567909+17.87375688j -17.87375688+17.86567909j]
 [  3.41439871 +3.422490

In [7]:
# make the Plist to contain matrices P0 and 1j*P0
Plist = [P0, 1j*P0]

QCQP = DenseSharedProjQCQP(Bmat, bVec, beta,
                            Umat, eVec, i0,
                             Plist, verbose = 1
                                )
# print fist elements of all inputs
# print(f"Bmat: {Bmat[0,0]}, bVec: {bVec[0]}, beta: {beta}, Umat: {Umat[0,0]}, eVec: {eVec[0] }")

t1 = time.time()
lags_init = np.zeros((2,))
lags_init[0] = -1
lags_init[0] = -0.988068084008008
lags_init[1] = -0.037771258012499
# New interface for specifying optimization hyperparameters. https://dolphindes.readthedocs.io/en/latest/api/dolphindes.cvxopt.OptimizationHyperparameters.html
opt_params = OptimizationHyperparameters(
    opttol=1e-12,                  
    gradConverge=True,           
    min_inner_iter=1,             
    max_restart=1,                
    penalty_ratio=1e-2,           
    penalty_reduction=0.1,        
    break_iter_period=5,          
    verbose=0)

y = - 2*(lags_init[0] * Plist[0].conj().T + lags_init[1]*Plist[1].conj().T) @ (-p/2)
# print(f"y: {y[:3]}")  # Debug print for yp and y
yp = (-S) @ (lags_init[0]*Plist[0] + lags_init[1]*Plist[1]) @ iVec
# print(f"yp: {yp[:3]}")  # Debug print for yp and y
cp = np.real(iVec.conj().T @ (y + yp))
print(f"Preconditioner contribution to dual constant term: {cp}")

A1 =  (-S) @ Plist[0]
A1sym = 0.5*(A1 + A1.conj().T)
# print(f"3x3 block of A1sym: {A1sym[:3,:3]}")  # Debug print for the symmetric part of the first projection constraint matrix
A2 =(-S) @ Plist[1]
A2sym = 0.5*(A2 + A2.conj().T)
# print(f"3x3 block of A2sym: {A2sym[:3,:3]}")  # Debug print for the symmetric part of the second projection constraint matrix
A = Bmat + lags_init[0]*(A1sym) + lags_init[1]*(A2sym)
# print(f"3x3 block of A: {A[:3,:3]}")  # Debug print for the Hessian matrix A
a = bVec - Fs @ lags_init
print(f"3 elements of S: {a[:3]}")  # Debug print for
x = np.linalg.solve(A, a)
# print(f"3 elements of x: {x[:3]}")  # Debug print for the solution x

xAx = x.conj().T @ A @ x
print(f"x^H A x: {xAx:.5e}")  # Debug print for the quadratic term in the dual function
cd = beta - cp
print(f"Constant term in dual function: {cd:.5e}")  # Debug print for the constant term in the dual function
dualVal = xAx + cd
# print(f"Dual value with initial lags: {dualVal}")  # Debug print for


# Note: 'newton' is usually faster than BFGS, both are prety fast with so few constraints.
result = QCQP.solve_current_dual_problem(method = 'bfgs', init_lags = lags_init, opt_params = opt_params)
print(f"bound: {result[0]:.4e}, time: {time.time()-t1}s, Lagrange multipliers: {QCQP.current_lags}")

3 rows of Fs: [[8.16444591e-03-8.07779690e-03j 3.57394360e+01-3.57395226e+01j]
 [8.17828272e-03-8.09163382e-03j 6.83688906e+00-6.83697571e+00j]
 [8.23257290e-03-8.14592398e-03j 4.29545319e+00-4.29553984e+00j]]
Precomputed 2 A matrices and Fs vectors.
Preconditioner contribution to dual constant term: 391630.31384334154
3 elements of S: [-1.35399491+1.35391257j -0.26231611+0.26223377j -0.16634938+0.16626703j]
x^H A x: 3.30933e+05+0.00000e+00j
Constant term in dual function: -3.30923e+05
x*^† A x*: 330932.9264859051
Preconditioner contribution to dual constant term: -391630.313843308
x*^† A x*: 5853801.380430479
Preconditioner contribution to dual constant term: -5914494.280680304
x*^† A x*: 4196940.377185507
Preconditioner contribution to dual constant term: -4257635.090629231
x*^† A x*: 3037137.817785459
Preconditioner contribution to dual constant term: -3097833.657593427
x*^† A x*: 2225276.1386125023
Preconditioner contribution to dual constant term: -2285972.6544683743
x*^† A x*: 16

KeyboardInterrupt: 